# 03 · Vector Databases
### Practical LLM Engineering — Module 03: Embeddings & Retrieval

> **Objectives**
> - What vector databases provide beyond a raw FAISS index
> - Implementing an in-memory vector DB with filtering and persistence
> - Overview of production DBs: Chroma, Qdrant, Pinecone, Weaviate, pgvector
> - Metadata filtering, namespaces, and collection management
> - Engineering: consistency, durability, and scaling patterns

---


## 1. Overview

A **vector database** extends a raw ANN index with:

```
Raw ANN index             Vector Database
──────────────────        ──────────────────────────────────────
• Fast k-NN search   +    • Metadata filtering (WHERE clause)
                          • Persistent storage (WAL / snapshots)
                          • CRUD operations (add / update / delete)
                          • Collections / namespaces
                          • Hybrid search (dense + sparse)
                          • Access control, replication, scaling
```

### Popular production vector databases

| DB | Open source | Hosting | Special strength |
|---|---|---|---|
| **Chroma** | ✅ | Self / Cloud | Simplest dev experience |
| **Qdrant** | ✅ | Self / Cloud | Payload filtering, performance |
| **Weaviate** | ✅ | Self / Cloud | Graph + vector hybrid |
| **Milvus** | ✅ | Self / Cloud | Billion-scale, GPU acceleration |
| **Pinecone** | ❌ | Cloud only | Fully managed, serverless |
| **pgvector** | ✅ | Postgres | SQL + vectors in one DB |
| **Redis VSS** | ✅ | Self / Cloud | Low latency, Redis ecosystem |


## 2. Setup

In [ ]:
# !pip install chromadb qdrant-client numpy matplotlib -q
import os, time, json, uuid, random
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional

print("✅ Libraries ready")

# ── Reuse MockEmbedder from notebook 01 ──────────────────────────────
from sklearn.preprocessing import normalize
class MockEmbedder:
    def __init__(self, dim=64, seed=42):
        self.dim   = dim
        self._proj = np.random.RandomState(seed).randn(10_000, dim).astype(np.float32)
    def _hash(self, w): return int(abs(hash(w.lower())) % 10_000)
    def embed_one(self, text):
        tokens = text.lower().split()
        if not tokens: return np.zeros(self.dim, dtype=np.float32)
        v = np.mean([self._proj[self._hash(t)] for t in tokens], axis=0)
        n = np.linalg.norm(v); return v / max(n, 1e-9)
    def embed(self, texts):
        return normalize(np.array([self.embed_one(t) for t in texts]))

embedder = MockEmbedder(dim=64)


## 3. Building an In-Memory Vector Database from Scratch

In [ ]:
from dataclasses import dataclass, field
from typing import Any

@dataclass
class VectorRecord:
    id:       str
    vector:   np.ndarray
    document: str
    metadata: dict = field(default_factory=dict)


class InMemoryVectorDB:
    """
    Feature-complete in-memory vector database.
    Supports: insert, update, delete, similarity search, metadata filtering,
              persistence (JSON snapshot), and basic collection management.
    """
    def __init__(self, name: str = "default", dim: int = 64):
        self.name       = name
        self.dim        = dim
        self._records:  dict[str, VectorRecord] = {}
        self._created   = time.strftime("%Y-%m-%dT%H:%M:%S")

    # ── CRUD ──────────────────────────────────────────────────────────
    def insert(self, doc_id: str, vector: np.ndarray,
               document: str, metadata: dict = None) -> str:
        self._records[doc_id] = VectorRecord(
            id=doc_id, vector=vector.astype(np.float32),
            document=document, metadata=metadata or {}
        )
        return doc_id

    def upsert(self, doc_id: str, vector: np.ndarray,
               document: str, metadata: dict = None):
        """Insert or update a record."""
        self.insert(doc_id, vector, document, metadata)

    def delete(self, doc_id: str) -> bool:
        return bool(self._records.pop(doc_id, None))

    def get(self, doc_id: str) -> Optional[VectorRecord]:
        return self._records.get(doc_id)

    def update_metadata(self, doc_id: str, metadata: dict):
        if doc_id in self._records:
            self._records[doc_id].metadata.update(metadata)

    # ── Search ────────────────────────────────────────────────────────
    def _filter_records(self, where: dict) -> list[VectorRecord]:
        """Apply metadata filter (simple equality + range checks)."""
        results = []
        for rec in self._records.values():
            match = True
            for key, condition in where.items():
                val = rec.metadata.get(key)
                if isinstance(condition, dict):
                    # Range operators: {"$gte": x, "$lte": y, "$eq": z}
                    for op, threshold in condition.items():
                        if op == "$gte" and not (val is not None and val >= threshold):
                            match = False
                        elif op == "$lte" and not (val is not None and val <= threshold):
                            match = False
                        elif op == "$eq" and val != threshold:
                            match = False
                        elif op == "$in" and val not in threshold:
                            match = False
                else:
                    if val != condition:
                        match = False
            if match:
                results.append(rec)
        return results

    def search(self, query_vector: np.ndarray, k: int = 5,
               where: dict = None) -> list[dict]:
        """
        Similarity search with optional metadata filtering.
        Returns list of {id, document, score, metadata}.
        """
        q = query_vector.astype(np.float32)
        q /= max(np.linalg.norm(q), 1e-9)

        candidates = self._filter_records(where) if where else list(self._records.values())
        if not candidates:
            return []

        vecs   = np.array([r.vector for r in candidates])
        scores = vecs @ q

        top_k_idx = np.argsort(-scores)[:k]
        return [
            {"id": candidates[i].id, "document": candidates[i].document,
             "score": float(scores[i]), "metadata": candidates[i].metadata}
            for i in top_k_idx
        ]

    # ── Persistence ───────────────────────────────────────────────────
    def save(self, path: str):
        data = {
            "name": self.name, "dim": self.dim,
            "records": {
                rid: {"id": r.id, "document": r.document,
                       "vector": r.vector.tolist(), "metadata": r.metadata}
                for rid, r in self._records.items()
            }
        }
        with open(path, "w") as f: json.dump(data, f)
        print(f"Saved {len(self._records)} records to {path}")

    @classmethod
    def load(cls, path: str) -> "InMemoryVectorDB":
        with open(path) as f: data = json.load(f)
        db = cls(name=data["name"], dim=data["dim"])
        for rid, r in data["records"].items():
            db.insert(rid, np.array(r["vector"], dtype=np.float32),
                       r["document"], r["metadata"])
        print(f"Loaded {len(db._records)} records from {path}")
        return db

    def __len__(self): return len(self._records)
    def __repr__(self): return f"InMemoryVectorDB(name={self.name!r}, n={len(self)})"


# ── Populate with documents ───────────────────────────────────────────
DOCUMENTS = [
    ("Neural networks learn by adjusting weights through backpropagation.",
     {"category": "ml", "level": "intro", "year": 2019}),
    ("Transformers revolutionised NLP with the self-attention mechanism.",
     {"category": "ml", "level": "intermediate", "year": 2017}),
    ("FAISS enables billion-scale similarity search on CPUs and GPUs.",
     {"category": "retrieval", "level": "advanced", "year": 2019}),
    ("HNSW is a graph-based ANN algorithm with excellent recall/speed.",
     {"category": "retrieval", "level": "advanced", "year": 2016}),
    ("Pasta carbonara uses eggs, pecorino romano, guanciale, and black pepper.",
     {"category": "cooking", "level": "intro", "year": 2020}),
    ("Rome was founded in 753 BC on the banks of the River Tiber.",
     {"category": "history", "level": "intro", "year": 2021}),
    ("LLMs are pre-trained on large corpora then fine-tuned with RLHF.",
     {"category": "ml", "level": "advanced", "year": 2022}),
    ("Qdrant provides filtered ANN search with payload conditions.",
     {"category": "retrieval", "level": "intermediate", "year": 2021}),
    ("Sourdough bread requires a live starter culture and long fermentation.",
     {"category": "cooking", "level": "intermediate", "year": 2020}),
    ("The Byzantine Empire was the continuation of the Roman Empire in the East.",
     {"category": "history", "level": "intermediate", "year": 2018}),
]

db = InMemoryVectorDB(name="knowledge-base", dim=embedder.dim)
for i, (text, meta) in enumerate(DOCUMENTS):
    vec = embedder.embed([text])[0]
    db.insert(f"doc_{i:03d}", vec, text, meta)

print(db)


## 4. Metadata Filtering

In [ ]:
# ── Filter + similarity search ────────────────────────────────────────
query  = "vector similarity search algorithms"
q_vec  = embedder.embed([query])[0]

print("Query:", query)
print()

# No filter
results_all = db.search(q_vec, k=3)
print("No filter (top 3):")
for r in results_all:
    print(f"  [{r['score']:.3f}] [{r['metadata']['category']}] {r['document'][:55]}")

print()

# Filter: category = retrieval
results_ret = db.search(q_vec, k=3, where={"category": "retrieval"})
print("Filter: category='retrieval':")
for r in results_ret:
    print(f"  [{r['score']:.3f}] {r['document'][:55]}")

print()

# Filter: year >= 2020 AND category = ml
results_recent = db.search(q_vec, k=5,
    where={"category": "ml", "year": {"$gte": 2020}})
print("Filter: category='ml' AND year >= 2020:")
for r in results_recent:
    print(f"  [{r['score']:.3f}] [{r['metadata']['year']}] {r['document'][:55]}")


## 5. Chroma and Qdrant: Production APIs

In [ ]:
# ── Chroma (client API demo) ──────────────────────────────────────────
CHROMA_DEMO = """
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()   # In-memory; use PersistentClient for disk

# Embedding function (wraps sentence-transformers)
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
    name="knowledge-base",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"}
)

# Add documents — Chroma handles embedding automatically
collection.add(
    documents=["Transformers use self-attention...", "FAISS enables fast search..."],
    metadatas=[{"category": "ml"}, {"category": "retrieval"}],
    ids=["doc_0", "doc_1"]
)

# Query with metadata filter
results = collection.query(
    query_texts=["attention mechanisms"],
    n_results=3,
    where={"category": "ml"}
)
"""

# ── Qdrant (client API demo) ──────────────────────────────────────────
QDRANT_DEMO = """
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue

client = QdrantClient(":memory:")

# Create collection
client.create_collection(
    collection_name="knowledge-base",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

# Upsert points
client.upsert(
    collection_name="knowledge-base",
    points=[
        PointStruct(id=1, vector=[...], payload={"category": "ml", "year": 2022}),
        PointStruct(id=2, vector=[...], payload={"category": "retrieval", "year": 2021}),
    ]
)

# Search with filter
results = client.search(
    collection_name="knowledge-base",
    query_vector=[...],
    query_filter=Filter(
        must=[FieldCondition(key="category", match=MatchValue(value="ml"))]
    ),
    limit=5,
)
"""

print("Chroma API pattern:")
print(CHROMA_DEMO)
print("─" * 60)
print("Qdrant API pattern:")
print(QDRANT_DEMO)


## 6. Persistence and CRUD Demo

In [ ]:
# ── Save / reload demonstration ───────────────────────────────────────
save_path = "/tmp/vector_db_snapshot.json"
db.save(save_path)
db2 = InMemoryVectorDB.load(save_path)

# Verify round-trip
q_vec = embedder.embed(["neural network training"])[0]
r1 = db.search(q_vec, k=2)
r2 = db2.search(q_vec, k=2)

print("Round-trip consistency check:")
for orig, loaded in zip(r1, r2):
    match = "✅" if abs(orig["score"] - loaded["score"]) < 1e-5 else "❌"
    print(f"  {match} {orig['id']}  score_orig={orig['score']:.4f}  score_loaded={loaded['score']:.4f}")

print()
# Update metadata
db.update_metadata("doc_000", {"reviewed": True, "quality": "high"})
rec = db.get("doc_000")
print(f"Updated metadata for doc_000: {rec.metadata}")

# Delete
deleted = db.delete("doc_009")
print(f"Deleted doc_009: {deleted}  (DB now has {len(db)} docs)")


## 7. Engineering Insights

### Choosing a vector database

```
Prototype / notebook:   ChromaDB (zero config, in-memory or file)
Production self-hosted: Qdrant (best performance + filtering)
Existing Postgres:      pgvector (no new infra needed)
Managed cloud:          Pinecone or Qdrant Cloud
Billion-scale:          Milvus (distributed, GPU-accelerated)
```

### pgvector: SQL + vectors in Postgres

```sql
-- Enable extension
CREATE EXTENSION vector;

-- Add vector column to existing table
ALTER TABLE documents ADD COLUMN embedding vector(768);

-- Create HNSW index
CREATE INDEX ON documents USING hnsw (embedding vector_cosine_ops);

-- Query: top-5 similar + metadata filter
SELECT id, content, embedding <=> $1 AS distance
FROM documents
WHERE category = 'ml' AND year >= 2022
ORDER BY distance
LIMIT 5;
```

### Key operational concerns

| Concern | Solution |
|---|---|
| Consistency | Write-ahead log; eventual consistency for replicas |
| Cold start | Pre-warm index before traffic |
| Schema migration | Add new metadata fields; re-index if model changes |
| Monitoring | Track p99 query latency, index fragmentation |
